In [3]:
# ReLU関数
import numpy as np

def relu(x):
    return np.maximum(x, 0)

def deriv_relu(x):
    return (x > 0).astype(x.dty)

In [4]:
# Sigmoid関数
def sigmoid(x):
    return np.exp(np.minimum(x, 0)) / (1 + np.exp(- np.abs(x)))

def deriv_sigmoid(x):
    return sigmoid(x) * (1 - sigmoid(x))

In [5]:
def np_log(x):
    return np.log(np.clip(x, 1e-10, 1e+10))

In [6]:
def train_xor(x, t, eps):
    """
    :param x: np.ndarray, input data, (batch_size, input_dim)
    :param t: np.ndarray, ground-truth labels, (batch_size, output_dim)
    :param eps: float, learning rate
    """
    global W1, b1, W2, b2

    batch_size = x.shape[0]

    # Forward propagation
    # np.matmul()：入力xと重みW1の行列積を計算
    # バイアスb1を足す
    u1 = np.matmul(x, W1) + b1  # (batch_size, hidden_dim)
    # 前の層で求めた値をReLU関数に通して、隠れ層の出力h1を求める
    h1 = relu(u1)

    # 隠れ層h1を入力として、重みW2とバイアスb2を使ったAffine変換を行う
    u2 = np.matmul(h1, W2) + b2  # (batch_size, output_dim)
    # シグモイド関数に通して、出力を確率に変換
    y = sigmoid(u2)


    # Compute loss
    # 2クラス交差エントロピーの計算
    cost = (- t * np_log(y) - (1 - t) * np_log(1 - y)).mean()

    # Backpropagation
    # シグモイド関数の逆伝播
    # 予測値から正解データを引く
    delta_2 = y - t  
    # ReLU関数の逆伝播
    # deriv_relu(u1)：ReLUの微分
    # W2.T：重みW2の転置行列
    delta_1 = deriv_relu(u1) * np.matmul(delta_2, W2.T)  

    # Compute gradients
    # 1層目の重みとバイアスの逆伝播
    dW1 = np.matmul(x.T, delta_1) / batch_size  
    db1 = np.matmul(np.ones(batch_size), delta_1) / batch_size   

    # 2層目の重みとバイアスの逆伝播
    dW2 = np.matmul(h1.T, delta_2) / batch_size  
    db2 = np.matmul(np.ones(batch_size), delta_2) / batch_size   

    # Update parameters
    # パラメータの更新
    W1 -= eps * dW1 
    b1 -= eps * db1 

    W2 -= eps * dW2 
    b2 -= eps * db2 

    return cost

def valid_xor(x, t):
    global W1, b1, W2, b2

    # Forward propagation
    u1 = np.matmul(x, W1) + b1
    h1 = relu(u1)

    u2 = np.matmul(h1, W2) + b2
    y = sigmoid(u2)

    # Compute loss
    cost = (- t * np_log(y) - (1 - t) * np_log(1 - y)).mean() 

    return cost, y